In [11]:
import pandas as pd



# Data Collection

## Grid intensity

Grid intensity is important for manufacturing, but it's still relevant for supply chains. Source: [Ember Energy](https://ember-energy.org/latest-insights/global-electricity-review-2025/major-countries-and-regions/)



In [ ]:
# gCO2/kWh - grid intensity is about CO2 released per unit of energy. Mostly about manufacturing, but still relevant
grid_intensity = {"china": 525, "mexico": 412, "s_korea": 390}


## Emission Factor
Ton-km emission factor is CO2 released per ton of commodity per kilometer transported. Relevant for transportation, depends on country's mix of transportation methods used. Because washing machines are transported mainly by trade vessels and trucks, these two are the main methods used in the calculation of the emission factor.

Lane-specific emission factors combine the IMO Fourth GHG Study global average with adjustments for typical vessel deployment on each lane (sourced from UNCTAD 2024 Chapter II) and feeder-megaship transshipment patterns documented in Notteboom & Rodrigue (2009).

Sources:
+ [US EPA SmartWay Carrier Emission Factors](epa.gov/smartway) - emission factor of Mexico-US trade routes
+ [New shipping routes highlight growing Asia-to-Mexico trade](https://www.freightwaves.com/news/new-shipping-routes-highlight-growing-asia-to-mexico-trade) - emission factor of Asia-Pacific trade routes
+ [Review of Maritime Transport 2024: Navigating Maritime Chokepoints](https://unctad.org/publication/review-maritime-transport-2024.) - GHG global average with country-specific adjustments for each trade route.
+ [Notteboom and Rodrigue](https://doi.org/10.1007/s10708-008-9210-4) - documents shipment patterns of feeder megaships

In [ ]:
# gCO2/ton-km  - emission factor is CO2 released per ton of commodity per kilometer transported. Different countries use different methods

ton_km = {
    "china": 4,
    "south_korea": 4.5, 
    "vietnam": 7, 
    "india": 6,
    "mexico": 80 
}

## Distance

This code calculates distances between ports for countries beyond the ocean(China, India, South Korea, Vietnam), as well as land distance over the US border for Mexico.

In [ ]:
import searoute as sr

# coordinates as [lon, lat]
route = sr.searoute([121.47, 31.23], [-118.27, 33.74])  # Shanghai → LA
distance_km = route.properties['length']  # in km, following real sea lanes

## US Census

Similar to UN Comtrade, but focuses on the USA and is more laconic.

[USA Trade Census](https://usatrade.census.gov/data/Perspective60/Browse/browsetables.aspx?utosid=5687ae9fc5be588295a68da15cd2f1cd&cache=tffv5e)
+ Data Source Selection: State Import Data(Harmonized System)
+ Filters:
    + Measures: Vessel SWT and Air SWT(kg) - The gross weight in kilograms of shipments made by seafaring vessel/airplane at customs
        + No data on land transportation there
    + State: All States
    + Commodity: 845011, 845012, 845019 and 845020(washing machines)
    + Country: India, South Korea, Mexico
    + Time: Jan 2025 - Mar 2026(monthly)

In [16]:
washing_machine_trade_filepath = "data/State Imports by HS Commodities_v4.csv"
washing_machine_df = pd.read_csv(washing_machine_trade_filepath, index_col=False, header=2)

In [17]:
washing_machine_df.head()

,Commodity,Country,Time,Air SWT (kg),Vessel SWT (kg),Unnamed: 5
0,845011 Washing Mach Automatic W Dry Line Cap N...,China,January 2025,174,"1,777,528",NaN
1,845011 Washing Mach Automatic W Dry Line Cap N...,China,February 2025,NaN,"2,459,827",NaN
2,845011 Washing Mach Automatic W Dry Line Cap N...,China,March 2025,NaN,"2,268,490",NaN
3,845011 Washing Mach Automatic W Dry Line Cap N...,China,April 2025,NaN,"2,512,291",NaN
4,845011 Washing Mach Automatic W Dry Line Cap N...,China,May 2025,NaN,"2,541,012",NaN


In [25]:
washing_machine_df_v1 = washing_machine_df.drop("Unnamed: 5", axis=1)
washing_machine_df_v1["Air SWT (kg)"] = washing_machine_df_v1["Air SWT (kg)"].str.replace(',', '')
washing_machine_df_v1["Vessel SWT (kg)"] = washing_machine_df_v1["Vessel SWT (kg)"].str.replace(',', '')

washing_machine_df_v1["Air SWT (kg)"] = pd.to_numeric(washing_machine_df_v1["Air SWT (kg)"])
washing_machine_df_v1["Vessel SWT (kg)"] = pd.to_numeric(washing_machine_df_v1["Vessel SWT (kg)"])

washing_machine_df_v1 = washing_machine_df_v1.fillna(0)

In [26]:
washing_machine_df_v1.head()

,Commodity,Country,Time,Air SWT (kg),Vessel SWT (kg)
0,845011 Washing Mach Automatic W Dry Line Cap N...,China,January 2025,174.0,1777528.0
1,845011 Washing Mach Automatic W Dry Line Cap N...,China,February 2025,0.0,2459827.0
2,845011 Washing Mach Automatic W Dry Line Cap N...,China,March 2025,0.0,2268490.0
3,845011 Washing Mach Automatic W Dry Line Cap N...,China,April 2025,0.0,2512291.0
4,845011 Washing Mach Automatic W Dry Line Cap N...,China,May 2025,0.0,2541012.0


In [ ]:
washing_machine_df_v1["Time"].values_count()

In [27]:
washing_machine_df_v1["Air SWT (kg)"].mean()

171.28148148148148